# PFE ML Final Colab Pipeline

This notebook is the final full-run version. It avoids smoke-test caps and writes durable outputs to Google Drive while using local Colab disk as a fast staging area.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone Or Refresh Repository

In [ ]:
REPO_DIR = '/content/pfein'
BACKEND_DIR = f'{REPO_DIR}/back_end'

%cd /content
!if [ ! -d "$REPO_DIR/.git" ]; then git clone --branch data-extraction --single-branch https://github.com/zribi1/pfein.git "$REPO_DIR"; else cd "$REPO_DIR" && git pull; fi
%cd $BACKEND_DIR

## 3. Configure Paths

In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/PFE ML Data/pfe_data'
WORK_DIR = '/content/pfe_work'

!mkdir -p "$DRIVE_ROOT" "$WORK_DIR"
!df -h /content
!ls -lah "$DRIVE_ROOT"

## 4. Install Colab Dependencies

In [ ]:
%cd $BACKEND_DIR
!pip install -q -r collabs/requirements-colab.txt

## 5. INPI Credentials

Run this cell only after replacing the placeholder values. Leave credentials out of Git and screenshots.

In [ ]:
import os

# Replace these values before the final INPI run.
os.environ['INPI_FTP_HOST'] = 'your_host'
os.environ['INPI_FTP_PORT'] = '21'
os.environ['INPI_FTP_USER'] = 'your_user'
os.environ['INPI_FTP_PASSWORD'] = 'your_password'
os.environ['INPI_FTP_PROTOCOL'] = 'ftp'
os.environ['INPI_REMOTE_BASE_DIR'] = '/'

## 6. Final Full Data Pipeline

This command downloads/exports INSEE, financials, INPI annual accounts and formalities, historical BODACC, builds clean tables, builds feature/label tables, trains the model, and writes the audit report. It uses no smoke-test caps.

In [ ]:
%cd $BACKEND_DIR

!python collabs/full_pipeline.py \
  --drive-root "$DRIVE_ROOT" \
  --work-dir "$WORK_DIR" \
  --repo-dir "$BACKEND_DIR" \
  --insee \
  --bilan \
  --inpi \
  --bodacc \
  --bodacc-mode historical \
  --bodacc-families PCL RCS-B \
  --bodacc-start-year 2008 \
  --bodacc-end-year 2025 \
  --start-year 2017 \
  --end-year 2025 \
  --train \
  --audit

## 7. Inspect Final Outputs

In [ ]:
!ls -lah "$DRIVE_ROOT/data-lake/features"
!ls -lah "$DRIVE_ROOT/ml-artifacts"
!ls -lah "$DRIVE_ROOT/reports"
!cat "$DRIVE_ROOT/ml-artifacts/model_metadata.json"
!head -80 "$DRIVE_ROOT/reports/data_lake_audit.md"

## 8. Resume Commands

If the runtime resets after downloads are already in Drive, rerun the setup cells, then rerun the final pipeline command. The scripts seed the local work directory from Drive and skip existing readable archives/manifests where possible.